# WEMoE NLP: Colab pipeline

Runs the full reproduction on a Colab GPU (a T4 is enough):

1. Setup: Drive, clone, install, unit tests
2. Optional smoke run (a few minutes, numbers are meaningless)
3. Experiment 1: five single-task BERT experts
4. Baselines: Simple Averaging and Task Arithmetic (lambda chosen on dev)
5. Experiment 2: WEMoE + FSFT (3 seeds), k sweep, intermediate-only ablation
6. Experiment 4: parameters, storage, latency
7. Summary tables

Checkpoints and results are written to Google Drive. Every step skips work that is already done,
so after a disconnect just rerun the cells.

If the repository is private, add a Colab secret named `GITHUB_TOKEN` (fine-grained, read-only).
Never paste a token into a cell.

In [13]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [14]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/wemoe-nlp'
PATHS = f'--set paths.checkpoint_dir={DRIVE_ROOT}/checkpoints --set paths.results_dir={DRIVE_ROOT}/results'
CFG = f'--config configs/default.yaml {PATHS}'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os
import subprocess

REPO = 'nguyenbaanh7779/wemoe-nlp'
BRANCH = 'main'
WORKDIR = '/content/wemoe-nlp'

if not os.path.exists(WORKDIR):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')  # only needed for a private repository
    except Exception:
        pass
    url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
    result = subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, url, WORKDIR],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError('git clone failed: ' + result.stderr.replace(token or '<no token>', '***'))
    # keep the token out of .git/config
    subprocess.run(['git', '-C', WORKDIR, 'remote', 'set-url', 'origin', f'https://github.com/{REPO}.git'],
                   check=True)

%cd {WORKDIR}
!git log --oneline -1
!pip install -q -r requirements.txt

/content/wemoe-nlp
891d681 (HEAD -> main, origin/main, origin/HEAD) Merge branch 'feat/wemoe-fsft-pipeline'


In [16]:
!python -m pytest -q

............................................                             [100%]
44 passed in 2.99s


## Optional: smoke run

Tiny subsets and a few steps, to check the whole pipeline end to end. Outputs stay on the VM.

In [17]:
SMOKE = '--config configs/smoke.yaml --set paths.checkpoint_dir=/content/smoke/checkpoints --set paths.results_dir=/content/smoke/results'
!python scripts/train_expert.py {SMOKE}
!python scripts/eval_baselines.py {SMOKE}
!python scripts/train_wemoe.py {SMOKE}
!python scripts/summarize_results.py --results-dir /content/smoke/results

[04:45:06] INFO root: Skipping mrpc: already trained (use --overwrite to retrain)
[04:45:06] INFO root: Skipping rte: already trained (use --overwrite to retrain)
[04:45:06] INFO root: Skipping cola: already trained (use --overwrite to retrain)
[04:45:06] INFO root: Skipping sst2: already trained (use --overwrite to retrain)
[04:45:06] INFO root: Skipping qnli: already trained (use --overwrite to retrain)
[04:45:14] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[04:45:15] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
[04:45:15] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
[04:45:15] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursiv

## Experiment 1: single-task experts

One recipe for every task (lr 2e-5, batch 32, 3 epochs, max length 128, fp16). The best epoch is
chosen on the dev split carved from the training set. About 40-60 minutes on a T4.

In [18]:
!python scripts/train_expert.py {CFG}

[04:46:01] INFO root: Skipping mrpc: already trained (use --overwrite to retrain)
[04:46:01] INFO root: Skipping rte: already trained (use --overwrite to retrain)
[04:46:01] INFO root: Skipping cola: already trained (use --overwrite to retrain)
[04:46:01] INFO root: Skipping sst2: already trained (use --overwrite to retrain)
[04:46:01] INFO root: Skipping qnli: already trained (use --overwrite to retrain)


## Baselines: Simple Averaging and Task Arithmetic

Task Arithmetic lambda is chosen on dev. WEMoE uses the same lambda to initialise its routers.

In [19]:
!python scripts/eval_baselines.py {CFG}

[04:46:15] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[04:46:15] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
[04:46:16] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
[04:46:16] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
[04:46:16] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
[04:46:16] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
[04:46:17] INFO datasets: TensorFlow 

## Experiment 2: WEMoE + FSFT

Routers only, cross-entropy on k=256 labelled examples per task from the few-shot pool (never seen by
the experts), best step on dev, a single evaluation on the official validation split.

In [20]:
for seed in [0, 1, 2]:
    !python scripts/train_wemoe.py {CFG} --set wemoe.seed={seed}

[04:47:52] INFO root: Skipping full-mean-k256-s0: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k256-s0 exists (use --overwrite to rerun)
[04:47:56] INFO root: Skipping full-mean-k256-s1: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k256-s1 exists (use --overwrite to rerun)
[04:48:00] INFO root: Skipping full-mean-k256-s2: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k256-s2 exists (use --overwrite to rerun)


## k sweep (Experiment 6) and MoE-scope ablation (intermediate only, as in the DANC code)

In [21]:
for k in [16, 64]:
    for seed in [0, 1, 2]:
        !python scripts/train_wemoe.py {CFG} --set wemoe.k={k} --set wemoe.seed={seed}

for seed in [0, 1, 2]:
    !python scripts/train_wemoe.py {CFG} --set wemoe.moe_scope=intermediate --set wemoe.seed={seed}

[04:48:04] INFO root: Skipping full-mean-k16-s0: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k16-s0 exists (use --overwrite to rerun)
[04:48:07] INFO root: Skipping full-mean-k16-s1: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k16-s1 exists (use --overwrite to rerun)
[04:48:11] INFO root: Skipping full-mean-k16-s2: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k16-s2 exists (use --overwrite to rerun)
[04:48:15] INFO root: Skipping full-mean-k64-s0: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k64-s0 exists (use --overwrite to rerun)
[04:48:19] INFO root: Skipping full-mean-k64-s1: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k64-s1 exists (use --overwrite to rerun)
[04:48:22] INFO root: Skipping full-mean-k64-s2: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k64-s2 exists (use --overwrite to rerun)
[04:48:26] INFO root: Skipping intermediate-mean-k256-s0: /content/drive/MyDrive/wemoe-nlp/results/wemoe/intermediate-

## Router ablation: input-independent routing (`router_hidden_layers=0`)

Each layer learns one fixed coefficient per expert (layer-wise merging, the same for every input), trained with the
same FSFT data. If WEMoE + FSFT beats it, the gain comes from input-dependent routing and not just from learning
better merge coefficients. With only 60 trainable numbers the default lr of 1e-4 barely moves them in 500 steps,
so this ablation uses lr 1e-3. Runs are named `full-mean-h0-k256-s<seed>`.

In [22]:
for seed in [0, 1, 2]:
    !python scripts/train_wemoe.py {CFG} --set wemoe.router_hidden_layers=0 --set wemoe.lr=1e-3 --set wemoe.seed={seed}

[04:48:37] INFO root: Skipping full-mean-h0-k256-s0: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-h0-k256-s0 exists (use --overwrite to rerun)
[04:48:41] INFO root: Skipping full-mean-h0-k256-s1: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-h0-k256-s1 exists (use --overwrite to rerun)
[04:48:45] INFO root: Skipping full-mean-h0-k256-s2: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-h0-k256-s2 exists (use --overwrite to rerun)


## Experiment 4: parameters, storage and latency

In [23]:
!python scripts/count_params.py {CFG} --latency

[04:48:52] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[04:48:52] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
Loading weights: 100% 199/199 [00:00<00:00, 15572.72it/s]
| Deployment | # checkpoints | Stored params | Size (fp32) | vs 5 models |
|---|---:|---:|---:|---:|
| 5 single-task models | 5 | 547.4M | 2.04 GiB | 100.0% |
| Task Arithmetic | 1 | 109.5M | 0.41 GiB | 20.0% |
| WEMoE (full MLP) | 1 | 400.0M | 1.49 GiB | 73.1% |
| WEMoE (intermediate only) | 1 | 258.4M | 0.96 GiB | 47.2% |

| Deployment | batch 1: ms/example (peak MB) | batch 32: ms/example (peak MB) |
|---|---:|---:|
| 5 single-task models | 11.64 (614) | 1.61 (704) |
| Task Arithmetic | 12.67 (435) | 1.63 (519) |
| WEMoE (full MLP) | 27.83 (1615) | 7.31 (1987) |
| WEMoE (intermediate only) | 18.90 (1061) | 4.69 (1387) |

Saved /content/drive/MyDrive/wemoe-nlp/results/efficiency.js

## Summary

In [24]:
!python scripts/summarize_results.py --results-dir {DRIVE_ROOT}/results

# Results (official GLUE validation splits)

| Model | MRPC (acc/F1) | RTE (acc) | CoLA (MCC) | SST-2 (acc) | QNLI (acc) | Avg score | Normalized | Gain over majority |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Single-task experts | 82.1/88.1 | 63.2 | 59.1 | 92.2 | 91.2 | 78.2 | 100.0 | 100.0 |
| Majority class | 68.4/81.2 | 52.7 | 0.0 | 50.9 | 50.5 | 45.8 | 56.4 | 0.0 |
| Simple Averaging (λ=1/5) | 68.4/81.2 | 46.9 | 23.3 | 55.4 | 65.9 | 53.3 | 66.8 | 6.6 |
| Task Arithmetic (λ=0.3) | 68.6/81.3 | 47.3 | 16.9 | 69.8 | 77.9 | 57.4 | 70.6 | 18.4 |
| WEMoE init (full, mean, h=0, λ=0.3, seeds=[0, 1, 2]) | 68.6±0.0/81.3±0.0 | 47.3±0.0 | 16.9±0.0 | 69.8±0.0 | 77.9±0.0 | 57.4±0.0 | 70.6±0.0 | 18.4±0.0 |
| WEMoE + FSFT (full, mean, h=0, k=256, lr=0.001, seeds=[0, 1, 2]) | 68.4±0.0/81.2±0.0 | 47.3±0.0 | 20.2±0.0 | 70.9±0.1 | 78.6±0.0 | 58.4±0.0 | 72.0±0.0 | 20.0±0.0 |
| WEMoE init (full, mean, λ=0.3, seeds=[0, 1, 2]) | 68.5±0.1/81.3±0.1 | 47.1±0.2 | 19.3±2.0 | 70.2±1.1 | 77.7±0.2 | 57.8±0